# Deployment: Object Detection Code for Railway Track Faults and Breaks

## System Performance Lab, Virginia Tech

**Goal:** Deploy the **robust, lightweight YOLO classifier** that flags model trains images for break or failure detection as either **OK** or **FAIL**.

**Organization of Python notebook.**
1. Install the packages
2. Upload the YOLO model
3. Deploy YOLO model on the test set


## 1. Install Packages

In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 35.8 MB/s eta 0:00:00


In [2]:
# load these packages
from ultralytics import YOLO
import os
import torch            # for version + CUDA check
import pandas as pd
import numpy as np
from PIL import Image   # only if you preview images
import matplotlib.pyplot as plt  # only if you preview images
import cv2, torch.nn as nn

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## 2. Upload the YOLO model

In [ ]:
m = YOLO('/runs/classify/train/weights/best.pt')

## 3. Deploy YOLO model on the test set

In [6]:
#  --- YOLO (cls) Eigen-CAM for a whole directory (no gradients needed) ---
import os, glob
import cv2, torch, numpy as np, matplotlib.pyplot as plt
import torch.nn as nn
from ultralytics import YOLO


# ---------------------------------------------------------
# 1. Build reusable Eigen-CAM function
# ---------------------------------------------------------
def eigen_cam(model_core, target_layer, bgr):
    H, W = bgr.shape[:2]

    # prep image
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    rgb_res = cv2.resize(rgb, (224, 224), interpolation=cv2.INTER_AREA).astype(np.float32)/255.0
    x = torch.from_numpy(rgb_res).permute(2,0,1).unsqueeze(0).to(next(model_core.parameters()).device)

    # hook storage
    acts = {}
    def fwd_hook(_, __, out): acts['y'] = out.detach()
    h = target_layer.register_forward_hook(fwd_hook)

    # forward pass (no grads)
    with torch.no_grad():
        _ = model_core(x)

    h.remove()

    # activations → Eigen-CAM
    A = acts['y'][0].cpu().numpy()               # [C,h,w]
    C, h0, w0 = A.shape
    M = A.reshape(C, -1)                         # [C, h*w]

    U, S, Vt = np.linalg.svd(M, full_matrices=False)
    pc1 = U[:, :1]
    cam = (M.T @ pc1).reshape(h0, w0)

    cam = np.maximum(cam, 0)
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    cam = cv2.resize(cam, (W, H))
    heat = cv2.applyColorMap((cam * 255).astype(np.uint8), cv2.COLORMAP_JET)
    overlay = cv2.addWeighted(bgr, 0.5, heat, 0.5, 0)

    return overlay, cam


# ---------------------------------------------------------
# 2. Process an entire directory of images
# ---------------------------------------------------------
def run_eigen_cam_batch(model, dir_path, save_dir=None, show=False):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    core = model.model.eval().to(device)

    # find last conv layer
    target_layer = next((L for L in reversed(list(core.modules())) if isinstance(L, nn.Conv2d)), None)
    assert target_layer, "No Conv2d layer found."

    # gather images
    exts = ['*.jpg','*.jpeg','*.png','*.bmp']
    files = [f for ext in exts for f in glob.glob(os.path.join(dir_path, ext))]
    print(f"Found {len(files)} images")

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)

    # process each image
    for img_path in files:
        print(f"Processing {os.path.basename(img_path)} ...")

        bgr = cv2.imread(img_path)
        if bgr is None:
            print("  ⚠️ Could not read image, skipping.")
            continue

        # YOLO classification prediction
        with torch.no_grad():
            pred = model.predict(source=img_path, imgsz=224, device=(0 if device=='cuda' else 'cpu'), verbose=False)
        annotated = pred[0].plot()

        # Eigen-CAM
        overlay, _ = eigen_cam(core, target_layer, bgr)

        # show results
        if show:
            plt.figure(figsize=(10,4))
            plt.subplot(1,2,1); plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)); plt.title("Prediction"); plt.axis('off')
            plt.subplot(1,2,2); plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB));  plt.title("Eigen-CAM focus"); plt.axis('off')
            plt.show()

        # save images
        if save_dir:
            base = os.path.basename(img_path)
            cv2.imwrite(os.path.join(save_dir, f"{base}_pred.jpg"), annotated)
            cv2.imwrite(os.path.join(save_dir, f"{base}_eigen.jpg"), overlay)

    print("Done.")


In [ ]:
test_dir = "/data/test/fail"
output_dir = "runs/classify/test/fail"

In [8]:
run_eigen_cam_batch(m, test_dir, save_dir=output_dir, show=False)

Found 64 images
Processing frame_000000.jpg ...
Processing frame_000001.jpg ...
Processing frame_000002.jpg ...
Processing frame_000003.jpg ...
Processing frame_000004.jpg ...
Processing frame_000005.jpg ...
Processing frame_000006.jpg ...
Processing frame_000007.jpg ...
Processing frame_000008.jpg ...
Processing frame_000009.jpg ...
Processing frame_000010.jpg ...
Processing frame_000011.jpg ...
Processing frame_000012.jpg ...
Processing frame_000013.jpg ...
Processing frame_000014.jpg ...
Processing frame_000015.jpg ...
Processing frame_000016.jpg ...
Processing frame_000017.jpg ...
Processing frame_000018.jpg ...
Processing frame_000019.jpg ...
Processing frame_000020.jpg ...
Processing frame_000021.jpg ...
Processing frame_000022.jpg ...
Processing frame_000023.jpg ...
Processing frame_000024.jpg ...
Processing frame_000025.jpg ...
Processing frame_000026.jpg ...
Processing frame_000027.jpg ...
Processing frame_000028.jpg ...
Processing frame_000029.jpg ...
Processing frame_000030.

In [ ]:
test_dir = "/data/test/ok"
output_dir = "/runs/classify/test/ok"

In [10]:
run_eigen_cam_batch(m, test_dir, save_dir=output_dir, show=False)

Found 54 images
Processing frame_000000.jpg ...
Processing frame_000001.jpg ...
Processing frame_000003.jpg ...
Processing frame_000002.jpg ...
Processing frame_000004.jpg ...
Processing frame_000005.jpg ...
Processing frame_000006.jpg ...
Processing frame_000008.jpg ...
Processing frame_000007.jpg ...
Processing frame_000009.jpg ...
Processing frame_000010.jpg ...
Processing frame_000011.jpg ...
Processing frame_000012.jpg ...
Processing frame_000013.jpg ...
Processing frame_000014.jpg ...
Processing frame_000015.jpg ...
Processing frame_000016.jpg ...
Processing frame_000018.jpg ...
Processing frame_000017.jpg ...
Processing frame_000019.jpg ...
Processing frame_000020.jpg ...
Processing frame_000021.jpg ...
Processing frame_000022.jpg ...
Processing frame_000023.jpg ...
Processing frame_000024.jpg ...
Processing frame_000025.jpg ...
Processing frame_000026.jpg ...
Processing frame_000027.jpg ...
Processing frame_000028.jpg ...
Processing frame_000031.jpg ...
Processing frame_000029.